In [1]:
from active_learning_sim2 import main

main()

ACTIVE LEARNING SIMULATION
  Repeats=10  Budget=80  Seed pool=40  Refit every=5
Loading brain delivery dataset...
  LC:    433 samples | 31 features | LC [0.0, 36.5]
  [PrepDist] Prep types before organic filter:
    ✓   188  'polymer-based nanoparticles'
    ✗   141  'free drug'
    ✓   101  'liposome'
    ✓    60  'solid lipid nanoparticles'
    ✗    53  'others'
    ✗    23  'gold nanoparticles'
    ✗    21  'emulsion'
    ✗    14  'albumin nanoparticles'
    ✗    13  'cationic liposome'
    ✗     9  'magnetic nanoparticles'
    ✗     7  'dendrimer'
    ✗     6  nan
    ✗     4  'magnetic liposome'
    ✗     4  'extracellular vesicles'
    ✗     4  'inorganic nanoparticles'
    ✗     2  'thermosensitive liposome'
    ✗     1  'micelle'
  [BrainFilter] Organic-only filter: 651 → 349 rows (302 inorganic/other rows removed)
  [OmicsFeatures] receptor → RECEPTOR_BBB_EXPR matches : ['GLUT-1', 'transferrin receptor', 'αvβ1', 'folic acid receptor', 'Lactoferrin recepter', 'low density lipo

In [2]:
import pandas as pd
import numpy as np
brain = pd.read_csv('datasetCELL2.csv')
print(brain['ID%'].describe())
print("\nTop 10 ID% values:")
print(brain['ID%'].nlargest(10))
print("\nFiltered to receptor mediated + transferrin:")
mask = brain['receptor'].str.contains('transferrin', na=False)
print(brain[mask]['ID%'].describe())

count    651.000000
mean       0.032976
std        0.094842
min        0.000000
25%        0.000505
50%        0.003200
75%        0.018550
max        1.000000
Name: ID%, dtype: float64

Top 10 ID% values:
1024    1.00000
582     0.85190
310     0.73930
957     0.66600
309     0.56360
1170    0.50400
436     0.48670
703     0.47322
62      0.47000
311     0.41980
Name: ID%, dtype: float64

Filtered to receptor mediated + transferrin:
count    17.000000
mean      0.018485
std       0.039789
min       0.000004
25%       0.000360
50%       0.002100
75%       0.023256
max       0.167000
Name: ID%, dtype: float64


In [4]:
import numpy as np

# 1. What does the log transform look like
brain['log_ID'] = np.log1p(brain['ID%'])
print(brain['log_ID'].describe())

# 2. How many training points are above log1p(0.1) = 0.095
print(f"Points above 0.1 %ID/g: {(brain['ID%'] > 0.1).sum()}")
print(f"Points above 0.5 %ID/g: {(brain['ID%'] > 0.5).sum()}")

# 3. What prep types dominate the top values
top = brain.nlargest(20, 'ID%')[['ID%','prep','receptor','animal-model','mechanism']]
print(top)

count    651.000000
mean       0.029190
std        0.075928
min        0.000000
25%        0.000505
50%        0.003195
75%        0.018380
max        0.693147
Name: log_ID, dtype: float64
Points above 0.1 %ID/g: 51
Points above 0.5 %ID/g: 6
           ID%                         prep receptor animal-model  \
1024  1.000000                    free drug      NaN       normal   
582   0.851900  polymer-based nanoparticles      NaN  brain tumor   
310   0.739300    solid lipid nanoparticles      NaN  brain tumor   
957   0.666000           gold nanoparticles      NaN  brain tumor   
309   0.563600    solid lipid nanoparticles      NaN  brain tumor   
1170  0.504000            cationic liposome      NaN  brain tumor   
436   0.486700  polymer-based nanoparticles      NaN       normal   
703   0.473220                       others      NaN       normal   
62    0.470000                     liposome      NaN       normal   
311   0.419800                    free drug      NaN  brain tumor   

In [6]:
# What does the filtered organic-only dataset look like
brain_filtered = brain[brain['prep'].isin([
    'polymer-based nanoparticles', 'liposome', 'solid lipid nanoparticles'
])]
print(brain_filtered.groupby(['mechanism','animal-model'])['ID%'].agg(['count','max','mean']))

                                                                  count  \
mechanism            animal-model                                         
absorption mediated  Alzheimer's disease                              0   
                     Parkinson's disease                              1   
                     brain tumor                                      8   
                     ischemic stroke                                  1   
                     ischemic stroke/reperfusion                      1   
                     normal                                          18   
                     others                                           1   
cell mediated        Alzheimer's disease                              0   
                     brain tumor                                      0   
                     normal                                           4   
                     others                                           0   
others               othe

In [6]:
#V, BM, CK, DG, EK
tfr = [1149.3, -0.424, -1.004, 0.210, -1.118]
slc2a1 = [4824.5, -0.463, -0.406, -0.210, -0.335]
ldlr = [78.7, -0.206, -0.599, -0.228, 0.316]
lrp1 = [160.7, 0.225, 1.317, -0.703, 0.526]
itgav = [36.9, -0.397, 0.478, -0.379, 0.650]
itgb3 = [4.4, 0.615, 1.844, 0.269, 3.694]
folr1 = [0]
insr = [577.4, -0.229, -0.379, -0.339, 0.211]
slc7a5 = [1160.3, -0.130, -0.841, -0.324, -0.766]
slc16a1 = [32.3, -0.627, -0.333, -0.195, 0.153]
slc5a7 = [0]
slc29a1 = [7.9, -0.127, 0.087, 0.091, 0.307]
slc22a5 = [24.9, 0.106, 0.171, -0.383, -0.414]
slc23a2 = [61.0, -0.007, 0.307, -0.399, 0.782]
